# Ekkubo — Eyes-Free Boda Navigation (Kampala)

Live OSM pipeline + Gemma 4 landmark disambiguation.

**Setup:** Add-ons → Secrets → `GEMINI_API_KEY` (Google AI Studio) → **toggle ON**.

**Important:** Toggle the secret ON *before* saving a version. Saved-version batch runs cannot read secrets added afterward. For the Gradio UI, use **Edit** session and run the last cell manually.


In [ ]:
!pip install -q requests gradio gTTS python-dotenv

In [ ]:
from pathlib import Path
import os

ROOT = Path('/kaggle/working')
pkg = ROOT / 'ekkubo'
pkg.mkdir(parents=True, exist_ok=True)
(ROOT / 'ekkubo' / '__init__.py').write_text('"""Ekkubo navigation."""\n', encoding='utf-8')
FILES = {"__init__.py": "\"\"\"Ekkubo navigation.\"\"\"\n", "config.py": "\"\"\"Shared configuration for Ekkubo navigation pipeline.\"\"\"\n\nimport os\nfrom pathlib import Path\n\n_PROJECT_ROOT = Path(__file__).resolve().parent.parent\n\n\ndef _load_dotenv() -> None:\n    try:\n        from dotenv import load_dotenv\n\n        load_dotenv(_PROJECT_ROOT / \".env\")\n    except ImportError:\n        pass\n\n\ndef _load_kaggle_secrets() -> None:\n    \"\"\"Load GEMINI_API_KEY from Kaggle notebook secrets when running on Kaggle.\"\"\"\n    if os.environ.get(\"GEMINI_API_KEY\") or os.environ.get(\"GOOGLE_API_KEY\"):\n        return\n    try:\n        from kaggle_secrets import UserSecretsClient\n\n        key = UserSecretsClient().get_secret(\"GEMINI_API_KEY\")\n        if key:\n            os.environ[\"GEMINI_API_KEY\"] = key\n    except Exception:\n        pass\n\n\n_load_dotenv()\n_load_kaggle_secrets()\n\n# --- App identity (required by Nominatim usage policy) ---\nAPP_NAME = \"Ekkubo\"\nAPP_VERSION = \"1.0\"\nUSER_AGENT = f\"{APP_NAME}/{APP_VERSION} (Kaggle Hackathon; eyes-free boda navigation, Kampala)\"\n\n# --- OpenStreetMap endpoints ---\nNOMINATIM_URL = \"https://nominatim.openstreetmap.org/search\"\nOSRM_URL = \"https://router.project-osrm.org/route/v1/driving\"\nOVERPASS_URL = \"https://overpass-api.de/api/interpreter\"\n\n# Kampala bounding box to bias geocoding (south, north, west, east)\nKAMPALA_VIEWBOX = (0.25, 0.38, 32.45, 32.75)\nKAMPALA_CENTER = (0.3476, 32.5825)\n\n# --- Rate limits ---\nNOMINATIM_MIN_INTERVAL_S = 1.0  # max 1 req/sec per Nominatim policy\nOVERPASS_MIN_INTERVAL_S = 1.0\n\n# --- Caching ---\nCACHE_DIR = Path(os.environ.get(\"EKKUBO_CACHE_DIR\", Path.home() / \".ekkubo_cache\"))\nCACHE_DB_PATH = CACHE_DIR / \"ekkubo_cache.db\"\nGEOCODE_TTL_S = 7 * 24 * 3600  # 7 days\nPOI_TTL_S = 24 * 3600  # 1 day\n\n# Coordinate rounding for cache keys (~11 m precision at equator)\nCOORD_ROUND_DIGITS = 4\nPOI_SEARCH_RADIUS_M = 100\n\n# --- Gemma 4 via Google Generative Language API ---\n# Set GEMINI_API_KEY or GOOGLE_API_KEY in environment / Kaggle secrets.\nGEMINI_API_KEY = os.environ.get(\"GEMINI_API_KEY\") or os.environ.get(\"GOOGLE_API_KEY\", \"\")\nGEMMA_MODEL = os.environ.get(\"EKKUBO_GEMMA_MODEL\", \"gemma-4-31b-it\")\nGEMINI_API_URL = (\n    f\"https://generativelanguage.googleapis.com/v1beta/models/{GEMMA_MODEL}:generateContent\"\n)\n\n# --- Speech ---\nWHISPER_MODEL = os.environ.get(\"EKKUBO_WHISPER_MODEL\", \"base\")\nTTS_LANG_PRIMARY = \"lg\"\nTTS_LANG_FALLBACK = \"en\"\n\n# Minimum expected TTS bytes per character (heuristic for broken output)\nTTS_MIN_BYTES_PER_CHAR = 0.5\n", "cache.py": "\"\"\"SQLite cache for Nominatim geocoding and Overpass POI queries.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport sqlite3\nimport time\nfrom contextlib import contextmanager\nfrom typing import Any, Generator\n\nfrom ekkubo.config import CACHE_DB_PATH, CACHE_DIR, COORD_ROUND_DIGITS\n\nlogger = logging.getLogger(__name__)\n\n\ndef round_coord(value: float) -> float:\n    return round(value, COORD_ROUND_DIGITS)\n\n\ndef make_geocode_key(query: str) -> str:\n    return f\"geocode:{query.strip().lower()}\"\n\n\ndef make_poi_key(lat: float, lon: float, poi_type: str, radius_m: int) -> str:\n    return f\"poi:{poi_type}:{round_coord(lat)}:{round_coord(lon)}:{radius_m}\"\n\n\nclass Cache:\n    \"\"\"TTL cache backed by SQLite.\"\"\"\n\n    def __init__(self, db_path: Path = CACHE_DB_PATH) -> None:\n        CACHE_DIR.mkdir(parents=True, exist_ok=True)\n        self.db_path = db_path\n        self._init_db()\n\n    def _init_db(self) -> None:\n        with self._connect() as conn:\n            conn.execute(\n                \"\"\"\n                CREATE TABLE IF NOT EXISTS cache (\n                    cache_key TEXT PRIMARY KEY,\n                    payload TEXT NOT NULL,\n                    created_at REAL NOT NULL,\n                    ttl_seconds REAL NOT NULL\n                )\n                \"\"\"\n            )\n            conn.commit()\n\n    @contextmanager\n    def _connect(self) -> Generator[sqlite3.Connection, None, None]:\n        conn = sqlite3.connect(self.db_path)\n        try:\n            yield conn\n        finally:\n            conn.close()\n\n    def get(self, key: str) -> Any | None:\n        now = time.time()\n        with self._connect() as conn:\n            row = conn.execute(\n                \"SELECT payload, created_at, ttl_seconds FROM cache WHERE cache_key = ?\",\n                (key,),\n            ).fetchone()\n        if not row:\n            logger.debug(\"cache MISS %s\", key)\n            return None\n        payload, created_at, ttl = row\n        if now - created_at > ttl:\n            logger.debug(\"cache EXPIRED %s\", key)\n            self.delete(key)\n            return None\n        logger.info(\"cache HIT %s\", key)\n        return json.loads(payload)\n\n    def set(self, key: str, value: Any, ttl_seconds: float) -> None:\n        now = time.time()\n        with self._connect() as conn:\n            conn.execute(\n                \"\"\"\n                INSERT OR REPLACE INTO cache (cache_key, payload, created_at, ttl_seconds)\n                VALUES (?, ?, ?, ?)\n                \"\"\",\n                (key, json.dumps(value), now, ttl_seconds),\n            )\n            conn.commit()\n        logger.debug(\"cache SET %s ttl=%ss\", key, ttl_seconds)\n\n    def delete(self, key: str) -> None:\n        with self._connect() as conn:\n            conn.execute(\"DELETE FROM cache WHERE cache_key = ?\", (key,))\n            conn.commit()\n", "geocoding.py": "\"\"\"Geocoding via Nominatim with rate limiting and caching.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport threading\nimport time\nfrom dataclasses import dataclass\nfrom typing import Any\n\nimport requests\n\nfrom ekkubo.cache import Cache, make_geocode_key\nfrom ekkubo.config import (\n    GEOCODE_TTL_S,\n    KAMPALA_VIEWBOX,\n    NOMINATIM_MIN_INTERVAL_S,\n    NOMINATIM_URL,\n    USER_AGENT,\n)\n\nlogger = logging.getLogger(__name__)\n\n_rate_lock = threading.Lock()\n_last_request_at = 0.0\n\n\n@dataclass\nclass GeocodeResult:\n    display_name: str\n    lat: float\n    lon: float\n    importance: float\n    raw: dict[str, Any]\n\n\nclass GeocodingError(Exception):\n    \"\"\"Raised when geocoding fails.\"\"\"\n\n\ndef _respect_rate_limit() -> None:\n    global _last_request_at\n    with _rate_lock:\n        elapsed = time.time() - _last_request_at\n        if elapsed < NOMINATIM_MIN_INTERVAL_S:\n            time.sleep(NOMINATIM_MIN_INTERVAL_S - elapsed)\n        _last_request_at = time.time()\n\n\ndef _nominatim_search(query: str, limit: int = 5) -> list[dict[str, Any]]:\n    south, north, west, east = KAMPALA_VIEWBOX\n    params = {\n        \"q\": query,\n        \"format\": \"json\",\n        \"limit\": limit,\n        \"viewbox\": f\"{west},{north},{east},{south}\",\n        \"bounded\": 1,\n        \"addressdetails\": 1,\n    }\n    headers = {\"User-Agent\": USER_AGENT}\n\n    _respect_rate_limit()\n    logger.info(\"Nominatim search: %r\", query)\n    resp = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=30)\n    if resp.status_code == 429:\n        raise GeocodingError(\"Nominatim rate limit hit \u2014 retry after backoff\")\n    resp.raise_for_status()\n    return resp.json()\n\n\ndef geocode(\n    query: str,\n    cache: Cache | None = None,\n    *,\n    use_cache: bool = True,\n) -> list[GeocodeResult]:\n    \"\"\"Geocode a place name to coordinates, preferring Kampala area.\"\"\"\n    cache = cache or Cache()\n    key = make_geocode_key(query)\n\n    if use_cache:\n        cached = cache.get(key)\n        if cached is not None:\n            return [GeocodeResult(**item) for item in cached]\n\n    try:\n        data = _nominatim_search(query)\n    except requests.RequestException as exc:\n        raise GeocodingError(f\"Nominatim request failed: {exc}\") from exc\n\n    results = [\n        GeocodeResult(\n            display_name=item.get(\"display_name\", query),\n            lat=float(item[\"lat\"]),\n            lon=float(item[\"lon\"]),\n            importance=float(item.get(\"importance\", 0)),\n            raw=item,\n        )\n        for item in data\n    ]\n\n    if use_cache and results:\n        cache.set(\n            key,\n            [\n                {\n                    \"display_name\": r.display_name,\n                    \"lat\": r.lat,\n                    \"lon\": r.lon,\n                    \"importance\": r.importance,\n                    \"raw\": r.raw,\n                }\n                for r in results\n            ],\n            GEOCODE_TTL_S,\n        )\n\n    return results\n\n\ndef geocode_best(\n    query: str,\n    cache: Cache | None = None,\n) -> GeocodeResult | None:\n    \"\"\"Return the highest-importance geocode result, or None.\"\"\"\n    results = geocode(query, cache=cache)\n    if not results:\n        return None\n    return max(results, key=lambda r: r.importance)\n", "routing.py": "\"\"\"Turn-by-turn routing via OSRM public demo server.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\nimport requests\n\nfrom ekkubo.config import OSRM_URL, USER_AGENT\n\nlogger = logging.getLogger(__name__)\n\n\n@dataclass\nclass RouteStep:\n    \"\"\"One maneuver along the route.\"\"\"\n\n    distance_m: float\n    duration_s: float\n    maneuver_type: str\n    modifier: str | None\n    name: str | None\n    lat: float\n    lon: float\n    instruction: str  # OSRM's default instruction text\n\n\n@dataclass\nclass Route:\n    distance_m: float\n    duration_s: float\n    steps: list[RouteStep] = field(default_factory=list)\n    geometry: list[list[float]] = field(default_factory=list)\n\n\nclass RoutingError(Exception):\n    \"\"\"Raised when OSRM cannot compute a route.\"\"\"\n\n\ndef _parse_step(step: dict[str, Any]) -> RouteStep:\n    maneuver = step.get(\"maneuver\", {})\n    loc = maneuver.get(\"location\", [0.0, 0.0])\n    return RouteStep(\n        distance_m=float(step.get(\"distance\", 0)),\n        duration_s=float(step.get(\"duration\", 0)),\n        maneuver_type=str(maneuver.get(\"type\", \"unknown\")),\n        modifier=maneuver.get(\"modifier\"),\n        name=step.get(\"name\") or None,\n        lon=float(loc[0]),\n        lat=float(loc[1]),\n        instruction=str(maneuver.get(\"instruction\", \"\")),\n    )\n\n\ndef get_route(\n    origin_lat: float,\n    origin_lon: float,\n    dest_lat: float,\n    dest_lon: float,\n) -> Route:\n    \"\"\"\n    Fetch a driving route from OSRM public demo server.\n\n    Production note: replace OSRM_URL with a self-hosted instance loaded with\n    Uganda's .osm.pbf extract from Geofabrik for reliability under load.\n    \"\"\"\n    coords = f\"{origin_lon},{origin_lat};{dest_lon},{dest_lat}\"\n    url = f\"{OSRM_URL}/{coords}\"\n    params = {\n        \"steps\": \"true\",\n        \"geometries\": \"geojson\",\n        \"overview\": \"full\",\n        \"annotations\": \"false\",\n    }\n    headers = {\"User-Agent\": USER_AGENT}\n\n    logger.info(\n        \"OSRM route: (%.4f, %.4f) -> (%.4f, %.4f)\",\n        origin_lat,\n        origin_lon,\n        dest_lat,\n        dest_lon,\n    )\n    try:\n        resp = requests.get(url, params=params, headers=headers, timeout=60)\n        resp.raise_for_status()\n        data = resp.json()\n    except requests.RequestException as exc:\n        raise RoutingError(f\"OSRM request failed: {exc}\") from exc\n\n    if data.get(\"code\") != \"Ok\" or not data.get(\"routes\"):\n        message = data.get(\"message\", \"No route found\")\n        raise RoutingError(f\"OSRM: {message}\")\n\n    route_data = data[\"routes\"][0]\n    legs = route_data.get(\"legs\", [])\n    steps: list[RouteStep] = []\n    for leg in legs:\n        for step in leg.get(\"steps\", []):\n            steps.append(_parse_step(step))\n\n    geometry: list[list[float]] = []\n    geom = route_data.get(\"geometry\", {})\n    if geom.get(\"type\") == \"LineString\":\n        geometry = geom.get(\"coordinates\", [])\n\n    return Route(\n        distance_m=float(route_data.get(\"distance\", 0)),\n        duration_s=float(route_data.get(\"duration\", 0)),\n        steps=steps,\n        geometry=geometry,\n    )\n", "landmarks.py": "\"\"\"POI lookup near route steps via Overpass API.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport threading\nimport time\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\nimport requests\n\nfrom ekkubo.cache import Cache, make_poi_key\nfrom ekkubo.config import (\n    OVERPASS_MIN_INTERVAL_S,\n    OVERPASS_URL,\n    POI_SEARCH_RADIUS_M,\n    POI_TTL_S,\n    USER_AGENT,\n)\n\nlogger = logging.getLogger(__name__)\n\n_rate_lock = threading.Lock()\n_last_request_at = 0.0\n\n# Amenity / shop tags we care about for Kampala boda navigation\nPOI_FILTERS = [\n    'node[\"amenity\"=\"fuel\"]',\n    'node[\"amenity\"=\"place_of_worship\"]',\n    'node[\"amenity\"=\"taxi\"]',\n    'node[\"amenity\"=\"marketplace\"]',\n    'node[\"shop\"]',\n    'way[\"amenity\"=\"fuel\"]',\n    'way[\"amenity\"=\"place_of_worship\"]',\n    'way[\"amenity\"=\"taxi\"]',\n    'way[\"amenity\"=\"marketplace\"]',\n    'way[\"shop\"]',\n]\n\n\n@dataclass\nclass POI:\n    osm_id: int\n    osm_type: str\n    name: str | None\n    amenity: str | None\n    shop: str | None\n    lat: float\n    lon: float\n    tags: dict[str, str] = field(default_factory=dict)\n\n    @property\n    def category(self) -> str:\n        if self.amenity:\n            return self.amenity\n        if self.shop:\n            return f\"shop:{self.shop}\"\n        return \"unknown\"\n\n    @property\n    def label(self) -> str:\n        if self.name:\n            return self.name\n        return self.category.replace(\"_\", \" \").title()\n\n\nclass OverpassError(Exception):\n    \"\"\"Raised when Overpass query fails.\"\"\"\n\n\ndef _respect_rate_limit() -> None:\n    global _last_request_at\n    with _rate_lock:\n        elapsed = time.time() - _last_request_at\n        if elapsed < OVERPASS_MIN_INTERVAL_S:\n            time.sleep(OVERPASS_MIN_INTERVAL_S - elapsed)\n        _last_request_at = time.time()\n\n\ndef _build_overpass_query(lat: float, lon: float, radius_m: int) -> str:\n    union = \"\\n\".join(\n        f\"  {f}(around:{radius_m},{lat},{lon});\" for f in POI_FILTERS\n    )\n    return f\"\"\"\n[out:json][timeout:25];\n(\n{union}\n);\nout center tags;\n\"\"\"\n\n\ndef _parse_element(element: dict[str, Any]) -> POI | None:\n    tags = element.get(\"tags\", {})\n    if element[\"type\"] == \"node\":\n        lat, lon = element.get(\"lat\"), element.get(\"lon\")\n    else:\n        center = element.get(\"center\", {})\n        lat, lon = center.get(\"lat\"), center.get(\"lon\")\n    if lat is None or lon is None:\n        return None\n    return POI(\n        osm_id=int(element[\"id\"]),\n        osm_type=str(element[\"type\"]),\n        name=tags.get(\"name\"),\n        amenity=tags.get(\"amenity\"),\n        shop=tags.get(\"shop\"),\n        lat=float(lat),\n        lon=float(lon),\n        tags=tags,\n    )\n\n\ndef query_pois_near(\n    lat: float,\n    lon: float,\n    cache: Cache | None = None,\n    *,\n    radius_m: int = POI_SEARCH_RADIUS_M,\n    use_cache: bool = True,\n) -> list[POI]:\n    \"\"\"Query real POIs within radius of a coordinate.\"\"\"\n    cache = cache or Cache()\n    key = make_poi_key(lat, lon, \"all\", radius_m)\n\n    if use_cache:\n        cached = cache.get(key)\n        if cached is not None:\n            return [POI(**item) for item in cached]\n\n    query = _build_overpass_query(lat, lon, radius_m)\n    headers = {\"User-Agent\": USER_AGENT}\n\n    _respect_rate_limit()\n    logger.info(\"Overpass POI query at (%.4f, %.4f) r=%dm\", lat, lon, radius_m)\n    try:\n        resp = requests.post(\n            OVERPASS_URL,\n            data={\"data\": query},\n            headers=headers,\n            timeout=35,\n        )\n        if resp.status_code == 429:\n            raise OverpassError(\"Overpass rate limit hit\")\n        resp.raise_for_status()\n        data = resp.json()\n    except requests.RequestException as exc:\n        raise OverpassError(f\"Overpass request failed: {exc}\") from exc\n\n    pois: list[POI] = []\n    for element in data.get(\"elements\", []):\n        poi = _parse_element(element)\n        if poi:\n            pois.append(poi)\n\n    if use_cache:\n        cache.set(\n            key,\n            [\n                {\n                    \"osm_id\": p.osm_id,\n                    \"osm_type\": p.osm_type,\n                    \"name\": p.name,\n                    \"amenity\": p.amenity,\n                    \"shop\": p.shop,\n                    \"lat\": p.lat,\n                    \"lon\": p.lon,\n                    \"tags\": p.tags,\n                }\n                for p in pois\n            ],\n            POI_TTL_S,\n        )\n\n    logger.info(\"Found %d POIs near (%.4f, %.4f)\", len(pois), lat, lon)\n    return pois\n\n\ndef pois_for_route_steps(\n    steps: list,\n    cache: Cache | None = None,\n    *,\n    radius_m: int = POI_SEARCH_RADIUS_M,\n) -> list[list[POI]]:\n    \"\"\"Fetch POI candidates for each route step maneuver point.\"\"\"\n    cache = cache or Cache()\n    return [\n        query_pois_near(step.lat, step.lon, cache=cache, radius_m=radius_m)\n        for step in steps\n    ]\n", "gemma_nav.py": "\"\"\"Gemma 4 integration for geocoding cleanup, landmark disambiguation, and instruction generation.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport re\nfrom typing import Any\n\nimport requests\n\nfrom ekkubo.config import GEMINI_API_KEY, GEMINI_API_URL\nfrom ekkubo.landmarks import POI\nfrom ekkubo.routing import RouteStep\n\nlogger = logging.getLogger(__name__)\n\n\nclass GemmaError(Exception):\n    \"\"\"Raised when Gemma API call fails or returns unusable output.\"\"\"\n\n\nSYSTEM_INSTRUCTION = \"\"\"You are Ekkubo, an audio navigation assistant for boda boda (motorcycle taxi)\nriders in Kampala, Uganda. Riders speak informally, often mixing Luganda and English.\n\nRules:\n- Keep spoken Luganda instructions SHORT (one sentence, max ~15 words) \u2014 riders hear this while moving.\n- Never invent landmarks not in the candidate list.\n- Prefer named POIs (fuel stations, mosques, markets, taxi stages) over generic descriptions.\n- If no good landmark exists, use street name or maneuver direction only.\n- Always respond with valid JSON only, no markdown fences.\"\"\"\n\n\ndef _call_gemma(user_prompt: str, *, temperature: float = 0.3) -> str:\n    if not GEMINI_API_KEY:\n        raise GemmaError(\n            \"GEMINI_API_KEY or GOOGLE_API_KEY not set. \"\n            \"Get one at https://aistudio.google.com/apikey\"\n        )\n\n    payload = {\n        \"systemInstruction\": {\"parts\": [{\"text\": SYSTEM_INSTRUCTION}]},\n        \"contents\": [{\"role\": \"user\", \"parts\": [{\"text\": user_prompt}]}],\n        \"generationConfig\": {\n            \"temperature\": temperature,\n        },\n    }\n    url = f\"{GEMINI_API_URL}?key={GEMINI_API_KEY}\"\n    try:\n        resp = requests.post(url, json=payload, timeout=90)\n        resp.raise_for_status()\n        data = resp.json()\n    except requests.RequestException as exc:\n        raise GemmaError(f\"Gemma API request failed: {exc}\") from exc\n\n    try:\n        return data[\"candidates\"][0][\"content\"][\"parts\"][0][\"text\"]\n    except (KeyError, IndexError) as exc:\n        raise GemmaError(f\"Unexpected Gemma response shape: {data}\") from exc\n\n\ndef _parse_json(text: str) -> Any:\n    text = text.strip()\n    text = re.sub(r\"^```(?:json)?\\s*\", \"\", text)\n    text = re.sub(r\"\\s*```$\", \"\", text)\n    try:\n        return json.loads(text)\n    except json.JSONDecodeError:\n        match = re.search(r\"\\{.*\\}\", text, re.DOTALL)\n        if match:\n            return json.loads(match.group())\n        raise\n\n\ndef extract_search_term(raw_request: str) -> str:\n    \"\"\"Use Gemma to clean informal destination phrasing before geocoding retry.\"\"\"\n    prompt = f\"\"\"The rider said this destination (possibly Luganda/English mix):\n\"{raw_request}\"\n\nExtract a clean place name suitable for OpenStreetMap geocoding in Kampala, Uganda.\nReturn JSON: {{\"search_term\": \"...\", \"confidence\": \"high|medium|low\"}}\"\"\"\n    raw = _call_gemma(prompt)\n    data = _parse_json(raw)\n    return str(data.get(\"search_term\", raw_request)).strip()\n\n\ndef suggest_clarifying_question(raw_request: str) -> str:\n    \"\"\"When geocoding fails, ask the rider a helpful follow-up.\"\"\"\n    prompt = f\"\"\"Geocoding failed for this rider request in Kampala:\n\"{raw_request}\"\n\nSuggest ONE short clarifying question to ask the rider (Luganda + English mix is fine).\nReturn JSON: {{\"question_luganda\": \"...\", \"question_english\": \"...\"}}\"\"\"\n    raw = _call_gemma(prompt, temperature=0.5)\n    data = _parse_json(raw)\n    lg = data.get(\"question_luganda\", \"\")\n    en = data.get(\"question_english\", \"\")\n    return f\"{lg}\\n({en})\" if lg and en else lg or en or \"Tukubuuza: oli wa? (Where exactly are you going?)\"\n\n\ndef _poi_to_dict(poi: POI) -> dict[str, Any]:\n    return {\n        \"name\": poi.name,\n        \"category\": poi.category,\n        \"label\": poi.label,\n        \"lat\": poi.lat,\n        \"lon\": poi.lon,\n    }\n\n\ndef disambiguate_step(\n    step: RouteStep,\n    candidate_pois: list[POI],\n    rider_phrasing_style: str = \"\",\n) -> dict[str, Any]:\n    \"\"\"\n    Pick the best landmark and generate bilingual spoken instructions for one step.\n\n    Returns dict with keys:\n      distance_m, maneuver, chosen_landmark, instruction_luganda, instruction_english\n    \"\"\"\n    poi_list = [_poi_to_dict(p) for p in candidate_pois[:15]]\n    prompt = f\"\"\"Route step for a boda boda rider in Kampala:\n- Distance to maneuver: {int(step.distance_m)} meters\n- Maneuver type: {step.maneuver_type}\n- Modifier: {step.modifier or \"none\"}\n- Street name: {step.name or \"unknown\"}\n- OSRM instruction: {step.instruction}\n- Rider's original style/phrasing: \"{rider_phrasing_style or \"informal Kampala mix\"}\"\n\nNearby POI candidates from OpenStreetMap (ONLY choose from this list or null):\n{json.dumps(poi_list, ensure_ascii=False)}\n\nTasks:\na) Pick the most locally-recognizable landmark from candidates (prefer named fuel, mosque, market, taxi stage)\nb) If none suitable, set chosen_landmark to null and use street/direction only\nc) Write SHORT spoken Luganda instruction + English display version\n\nReturn JSON:\n{{\n  \"distance_m\": {int(step.distance_m)},\n  \"maneuver\": \"{step.maneuver_type}\",\n  \"chosen_landmark\": \"name or null\",\n  \"instruction_luganda\": \"...\",\n  \"instruction_english\": \"...\"\n}}\"\"\"\n    raw = _call_gemma(prompt)\n    result = _parse_json(raw)\n    result.setdefault(\"distance_m\", int(step.distance_m))\n    result.setdefault(\"maneuver\", step.maneuver_type)\n    result.setdefault(\"chosen_landmark\", None)\n    result.setdefault(\"instruction_luganda\", step.instruction)\n    result.setdefault(\"instruction_english\", step.instruction)\n    return result\n\n\ndef disambiguate_route(\n    steps: list[RouteStep],\n    pois_per_step: list[list[POI]],\n    rider_request: str = \"\",\n) -> list[dict[str, Any]]:\n    \"\"\"Run landmark disambiguation for every route step.\"\"\"\n    instructions: list[dict[str, Any]] = []\n    for step, pois in zip(steps, pois_per_step):\n        try:\n            instr = disambiguate_step(step, pois, rider_phrasing_style=rider_request)\n        except GemmaError as exc:\n            logger.warning(\"Gemma failed for step, using fallback: %s\", exc)\n            instr = {\n                \"distance_m\": int(step.distance_m),\n                \"maneuver\": step.maneuver_type,\n                \"chosen_landmark\": None,\n                \"instruction_luganda\": step.instruction,\n                \"instruction_english\": step.instruction,\n            }\n        instructions.append(instr)\n    return instructions\n", "speech.py": "\"\"\"Speech I/O: Whisper STT and gTTS with Luganda fallback.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport tempfile\nfrom pathlib import Path\n\nfrom ekkubo.config import TTS_LANG_FALLBACK, TTS_LANG_PRIMARY, TTS_MIN_BYTES_PER_CHAR, WHISPER_MODEL\n\nlogger = logging.getLogger(__name__)\n\n_whisper_model = None\n\n\ndef _get_whisper():\n    global _whisper_model\n    if _whisper_model is None:\n        import whisper\n\n        logger.info(\"Loading Whisper model: %s\", WHISPER_MODEL)\n        _whisper_model = whisper.load_model(WHISPER_MODEL)\n    return _whisper_model\n\n\ndef transcribe_audio(audio_path: str | Path) -> str:\n    \"\"\"Transcribe rider speech with Whisper.\"\"\"\n    model = _get_whisper()\n    result = model.transcribe(str(audio_path), fp16=False)\n    text = (result.get(\"text\") or \"\").strip()\n    logger.info(\"Whisper transcription: %r\", text)\n    return text\n\n\ndef synthesize_speech(text: str, output_path: str | Path | None = None) -> Path:\n    \"\"\"\n    Generate spoken audio with gTTS (Luganda primary, English fallback per phrase).\n\n    Returns path to MP3 file.\n    \"\"\"\n    from gtts import gTTS\n\n    if not text.strip():\n        raise ValueError(\"Cannot synthesize empty text\")\n\n    if output_path is None:\n        fd, tmp = tempfile.mkstemp(suffix=\".mp3\")\n        import os\n\n        os.close(fd)\n        output_path = Path(tmp)\n    else:\n        output_path = Path(output_path)\n\n    lang = TTS_LANG_PRIMARY\n    try:\n        tts = gTTS(text=text, lang=lang)\n        tts.save(str(output_path))\n        size = output_path.stat().st_size\n        min_expected = int(len(text) * TTS_MIN_BYTES_PER_CHAR)\n        if size < min_expected:\n            raise RuntimeError(\n                f\"gTTS output suspiciously small ({size} bytes for {len(text)} chars)\"\n            )\n        logger.info(\"gTTS (%s) OK: %d bytes\", lang, size)\n    except Exception as exc:\n        logger.warning(\"gTTS %s failed (%s), falling back to %s\", lang, exc, TTS_LANG_FALLBACK)\n        tts = gTTS(text=text, lang=TTS_LANG_FALLBACK)\n        tts.save(str(output_path))\n\n    return output_path\n\n\ndef concatenate_instructions_luganda(instructions: list[dict]) -> str:\n    \"\"\"Build one spoken script from route step Luganda instructions.\"\"\"\n    parts = [i.get(\"instruction_luganda\", \"\") for i in instructions if i.get(\"instruction_luganda\")]\n    return \". \".join(parts)\n", "pipeline.py": "\"\"\"Orchestrates the full Ekkubo navigation pipeline.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport time\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom typing import Callable\n\nfrom ekkubo.cache import Cache\nfrom ekkubo.geocoding import GeocodeResult, GeocodingError, geocode, geocode_best\nfrom ekkubo.gemma_nav import (\n    GemmaError,\n    disambiguate_route,\n    extract_search_term,\n    suggest_clarifying_question,\n)\nfrom ekkubo.landmarks import OverpassError, pois_for_route_steps\nfrom ekkubo.routing import Route, RoutingError, get_route\nfrom ekkubo.speech import concatenate_instructions_luganda, synthesize_speech\n\nlogger = logging.getLogger(__name__)\n\nStatusCallback = Callable[[str], None]\n\n\nclass PipelineStatus(str, Enum):\n    IDLE = \"idle\"\n    GEOCODING = \"geocoding\"\n    ROUTING = \"routing\"\n    FINDING_LANDMARKS = \"finding landmarks\"\n    GENERATING_INSTRUCTIONS = \"generating instructions\"\n    GENERATING_SPEECH = \"generating speech\"\n    DONE = \"done\"\n    ERROR = \"error\"\n    CLARIFY = \"clarify\"\n\n\n@dataclass\nclass NavigationResult:\n    status: PipelineStatus\n    origin: GeocodeResult | None = None\n    destination: GeocodeResult | None = None\n    route: Route | None = None\n    instructions: list[dict] = field(default_factory=list)\n    audio_path: str | None = None\n    message: str = \"\"\n    clarifying_question: str | None = None\n\n\ndef _notify(cb: StatusCallback | None, status: str) -> None:\n    logger.info(\"status: %s\", status)\n    if cb:\n        cb(status)\n\n\ndef _geocode_with_retry(\n    query: str,\n    cache: Cache,\n    raw_request: str,\n    on_status: StatusCallback | None,\n) -> GeocodeResult | None:\n    _notify(on_status, PipelineStatus.GEOCODING.value)\n    result = geocode_best(query, cache=cache)\n    if result:\n        return result\n\n    # Retry with Gemma-cleaned search term\n    try:\n        cleaned = extract_search_term(raw_request)\n        if cleaned.lower() != query.lower():\n            logger.info(\"Retrying geocode with cleaned term: %r\", cleaned)\n            time.sleep(1)  # respect Nominatim rate limit between attempts\n            result = geocode_best(cleaned, cache=cache)\n            if result:\n                return result\n    except GemmaError as exc:\n        logger.warning(\"Gemma search-term extraction failed: %s\", exc)\n\n    return None\n\n\ndef navigate(\n    origin_query: str,\n    destination_query: str,\n    *,\n    cache: Cache | None = None,\n    on_status: StatusCallback | None = None,\n    generate_audio: bool = True,\n) -> NavigationResult:\n    \"\"\"\n    Full live pipeline: geocode \u2192 route \u2192 POIs \u2192 Gemma instructions \u2192 TTS.\n\n    All landmark data comes from live Nominatim/Overpass \u2014 nothing hardcoded.\n    \"\"\"\n    cache = cache or Cache()\n    raw_dest = destination_query\n\n    try:\n        origin = _geocode_with_retry(origin_query, cache, origin_query, on_status)\n        if not origin:\n            question = _safe_clarify(origin_query)\n            return NavigationResult(\n                status=PipelineStatus.CLARIFY,\n                message=f\"Could not find starting point: {origin_query}\",\n                clarifying_question=question,\n            )\n\n        dest = _geocode_with_retry(destination_query, cache, raw_dest, on_status)\n        if not dest:\n            question = _safe_clarify(raw_dest)\n            return NavigationResult(\n                status=PipelineStatus.CLARIFY,\n                message=f\"Could not find destination: {destination_query}\",\n                clarifying_question=question,\n            )\n\n        _notify(on_status, PipelineStatus.ROUTING.value)\n        route = get_route(origin.lat, origin.lon, dest.lat, dest.lon)\n\n        _notify(on_status, PipelineStatus.FINDING_LANDMARKS.value)\n        try:\n            pois_per_step = pois_for_route_steps(route.steps, cache=cache)\n        except OverpassError as exc:\n            logger.warning(\"Overpass failed, continuing with empty POI lists: %s\", exc)\n            pois_per_step = [[] for _ in route.steps]\n\n        _notify(on_status, PipelineStatus.GENERATING_INSTRUCTIONS.value)\n        instructions = disambiguate_route(route.steps, pois_per_step, rider_request=raw_dest)\n\n        audio_path = None\n        if generate_audio and instructions:\n            _notify(on_status, PipelineStatus.GENERATING_SPEECH.value)\n            script = concatenate_instructions_luganda(instructions)\n            audio_path = str(synthesize_speech(script))\n\n        _notify(on_status, PipelineStatus.DONE.value)\n        return NavigationResult(\n            status=PipelineStatus.DONE,\n            origin=origin,\n            destination=dest,\n            route=route,\n            instructions=instructions,\n            audio_path=audio_path,\n            message=f\"Route ready: {len(instructions)} steps, \"\n            f\"{route.distance_m / 1000:.1f} km, ~{route.duration_s / 60:.0f} min\",\n        )\n\n    except RoutingError as exc:\n        _notify(on_status, PipelineStatus.ERROR.value)\n        return NavigationResult(\n            status=PipelineStatus.ERROR,\n            message=f\"No route found: {exc}\",\n        )\n    except GeocodingError as exc:\n        _notify(on_status, PipelineStatus.ERROR.value)\n        return NavigationResult(\n            status=PipelineStatus.ERROR,\n            message=str(exc),\n        )\n    except Exception as exc:\n        logger.exception(\"Pipeline failed\")\n        _notify(on_status, PipelineStatus.ERROR.value)\n        return NavigationResult(\n            status=PipelineStatus.ERROR,\n            message=f\"Unexpected error: {exc}\",\n        )\n\n\ndef _safe_clarify(raw_request: str) -> str:\n    try:\n        return suggest_clarifying_question(raw_request)\n    except GemmaError:\n        return \"Tukubuuza: oli wa ddala? (Can you describe the place more clearly?)\"\n", "app.py": "\"\"\"Gradio UI for Ekkubo \u2014 eyes-free boda navigation in Kampala.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nfrom pathlib import Path\n\nimport gradio as gr\n\nfrom ekkubo.pipeline import NavigationResult, PipelineStatus, navigate\nfrom ekkubo.speech import synthesize_speech, transcribe_audio\n\nlogging.basicConfig(level=logging.INFO, format=\"%(asctime)s %(name)s %(message)s\")\nlogger = logging.getLogger(__name__)\n\n\ndef _format_steps(result: NavigationResult) -> str:\n    if not result.instructions:\n        return result.message or \"No instructions yet.\"\n\n    lines = [\n        f\"**{result.message}**\",\n        f\"From: {result.origin.display_name if result.origin else '?'}\",\n        f\"To: {result.destination.display_name if result.destination else '?'}\",\n        \"\",\n    ]\n    for i, step in enumerate(result.instructions, 1):\n        landmark = step.get(\"chosen_landmark\") or \"(no landmark \u2014 street/direction only)\"\n        lines.append(f\"### Step {i} \u2014 {step.get('distance_m', '?')} m\")\n        lines.append(f\"**Landmark:** {landmark}\")\n        lines.append(f\"**Luganda:** {step.get('instruction_luganda', '')}\")\n        lines.append(f\"**English:** {step.get('instruction_english', '')}\")\n        lines.append(\"\")\n    return \"\\n\".join(lines)\n\n\ndef run_navigation(\n    origin_text: str,\n    dest_text: str,\n    audio_input,\n    status_state: str,\n) -> tuple[str, str, str | None, str]:\n    \"\"\"Gradio callback \u2014 supports text or mic input for destination.\"\"\"\n    dest = (dest_text or \"\").strip()\n    if audio_input is not None:\n        try:\n            dest = transcribe_audio(audio_input) or dest\n        except Exception as exc:\n            logger.warning(\"STT failed: %s\", exc)\n\n    origin = (origin_text or \"\").strip()\n    if not origin or not dest:\n        return \"Enter both origin and destination.\", status_state, None, PipelineStatus.ERROR.value\n\n    statuses: list[str] = []\n\n    def on_status(s: str) -> None:\n        statuses.append(s)\n\n    result = navigate(origin, dest, on_status=on_status, generate_audio=True)\n    status = \" \u2192 \".join(statuses) if statuses else result.status.value\n\n    if result.status == PipelineStatus.CLARIFY:\n        text = f\"**Need clarification**\\n\\n{result.clarifying_question or result.message}\"\n        return text, status, None, result.status.value\n\n    if result.status == PipelineStatus.ERROR:\n        return f\"**Error:** {result.message}\", status, None, result.status.value\n\n    return _format_steps(result), status, result.audio_path, result.status.value\n\n\ndef build_app() -> gr.Blocks:\n    with gr.Blocks(title=\"Ekkubo \u2014 Eyes-Free Boda Navigation\") as app:\n        gr.Markdown(\n            \"\"\"\n            # Ekkubo\n            **Real eyes-free audio navigation for boda boda riders in Kampala.**\n\n            Live data from OpenStreetMap (Nominatim + OSRM + Overpass).\n            Landmark disambiguation powered by Gemma 4.\n            \"\"\"\n        )\n\n        with gr.Row():\n            origin = gr.Textbox(\n                label=\"Starting point\",\n                placeholder=\"e.g. Makerere University, Kampala\",\n                value=\"Makerere University, Kampala\",\n            )\n            dest = gr.Textbox(\n                label=\"Destination (text)\",\n                placeholder=\"e.g. Kisaasi, Kampala\",\n            )\n\n        mic = gr.Audio(sources=[\"microphone\"], type=\"filepath\", label=\"Or speak destination\")\n        status = gr.Textbox(label=\"Pipeline status\", interactive=False, value=\"idle\")\n        btn = gr.Button(\"Get directions\", variant=\"primary\")\n\n        output = gr.Markdown(label=\"Route steps\")\n        audio_out = gr.Audio(label=\"Spoken route (Luganda)\", type=\"filepath\")\n\n        btn.click(\n            fn=run_navigation,\n            inputs=[origin, dest, mic, status],\n            outputs=[output, status, audio_out, status],\n        )\n\n        gr.Markdown(\n            \"\"\"\n            ---\n            **Architecture:** Nominatim (geocode) \u2192 OSRM public demo (routing) \u2192\n            Overpass (POIs) \u2192 Gemma 4 (landmark pick + Luganda instructions) \u2192 gTTS.\n\n            *Production:* self-host OSRM with Uganda OSM extract from Geofabrik for reliability.\n            \"\"\"\n        )\n\n    return app\n\n\ndef main() -> None:\n    app = build_app()\n    app.launch(server_name=\"0.0.0.0\", server_port=7860)\n\n\nif __name__ == \"__main__\":\n    main()\n"}
for name, content in FILES.items():
    path = pkg / name
    path.write_text(content, encoding='utf-8')
    print('wrote', path)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import logging
import os
import sys

logging.basicConfig(level=logging.INFO)
sys.path.insert(0, "/kaggle/working")


def _ensure_gemini_key() -> bool:
    """Load GEMINI_API_KEY from env or Kaggle Secrets. Returns True if set."""
    if os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"):
        print("GEMINI_API_KEY: loaded from environment")
        return True
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
        print("GEMINI_API_KEY: loaded from Kaggle Secrets")
        return True
    except Exception as exc:
        print("WARNING: could not load GEMINI_API_KEY:", type(exc).__name__, exc)
        print(
            "Saved-version runs need the secret toggled ON *before* Save Version. "
            "For full Gemma output: Edit session → Secrets ON → Run All."
        )
        return False


has_key = _ensure_gemini_key()

from ekkubo.pipeline import navigate, PipelineStatus

statuses = []

def on_status(s):
    statuses.append(s)
    print("status:", s)

print("Running live pipeline: Makerere -> Kisaasi")
result = navigate(
    "Makerere University, Kampala",
    "Kisaasi, Kampala",
    on_status=on_status,
    generate_audio=has_key,
)

print("\n=== RESULT ===")
print("status:", result.status)
print("message:", result.message)
for i, step in enumerate(result.instructions[:5], 1):
    print(f"\nStep {i} ({step.get('distance_m')}m)")
    print("  landmark:", step.get("chosen_landmark"))
    print("  LG:", step.get("instruction_luganda"))
    print("  EN:", step.get("instruction_english"))
if len(result.instructions) > 5:
    print(f"... +{len(result.instructions) - 5} more steps")
if result.audio_path:
    print("\nAudio:", result.audio_path)

## Interactive demo (run cell below manually — Gradio blocks the kernel)

In [ ]:
import os
import sys

sys.path.insert(0, "/kaggle/working")

# Batch/saved-version runs cannot host Gradio — interactive Edit session only.
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch":
    print("Skipping Gradio (batch run). Open Edit session and run this cell for the UI demo.")
else:
    from ekkubo.app import build_app

    app = build_app()
    app.launch(server_name="0.0.0.0", server_port=7860, debug=True, share=False)